# 원본 프레임에서 가까운 쪽 관절 찍기

탭마다 선수의 **어깨·골반·무릎·발목**을 독립적으로 판독합니다. 사진에는 내가 찍은 점만 나타납니다. 가까운 쪽은 화면의 좌우가 아니라 선수 본인의 왼쪽 또는 오른쪽을 뜻합니다.

1. 영상 탭의 선수 방향을 확인하고 관절 버튼을 고릅니다. 한 위치를 자신 있게 짚을 수 있으면 원본 사진을 한 번 클릭하세요. 점과 원본 픽셀 좌표가 즉시 저장됩니다. 같은 관절을 다시 클릭하면 위치를 수정할 수 있습니다.
2. 원판이나 옷에 가려 기준점이 보이지 않으면 **가려서 판독 불가**를 누르세요. 일부가 보이더라도 어디를 찍어야 할지 확신이 없으면 **위치 불확실**을 누르세요. 두 경우 모두 좌표를 비워 두고 사유를 저장합니다. 아무 조작도 하지 않은 관절은 **미검수**로 남습니다.
3. **원본 크기로 확대**하면 작은 부위를 스크롤해서 확인할 수 있습니다. **다시 읽기** 또는 다음 실행에서 저장된 판독을 다시 볼 수 있습니다.

가려졌지만 근거를 두고 위치를 추정할 수 있으면 클릭한 뒤 **가림 추정점으로 변경**을 누릅니다. 이미 찍은 점도 다시 클릭할 필요 없이 상태만 바꿀 수 있습니다. 마름모는 추정점이고 십자는 일반 클릭점입니다. 판단 근거가 부족하면 좌표 없이 판독 불가로 둡니다.

**관절별 기준**은 발목의 복사뼈(malleolus), 무릎의 관절선(joint line), 골반의 고관절 소켓(hip socket)입니다. c04 어깨는 견봉(acromion)을 기준으로 찍었습니다. 옷이 윤곽을 가려 한 점으로 좁히기 어려울 수 있습니다. 골반을 클릭하면 소켓의 화면상 위치를 추정한 점으로 저장합니다.

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from wlcv.annotation.pose_points import pose_point_editor
from wlcv.config import ROOT, SPLIT, VIDEOS

samples = [("a03_sn_f_01", 19), ("b02_sn_h_02", 36), ("c04_sn_f_01", 78)]

split = pd.read_csv(SPLIT)
validation = split.loc[split["role"] == "validation", ["video_id", "person_id", "role"]]
metadata = pd.read_csv(ROOT / "data/metadata/videos.csv")
videos = validation.merge(metadata[["video_id", "filename", "near_side"]], on="video_id",
                          validate="one_to_one").set_index("video_id")

In [ ]:
editors = []
for video, frame in samples:
    if video not in videos.index:
        raise ValueError(f"{video}가 기존 검증 분할에 있는지 확인하세요.")
    info = videos.loc[video]
    editor = pose_point_editor(VIDEOS / info.filename, frame, info.person_id,
                               info.near_side, split=info.role)
    editors.append(editor)

tabs = widgets.Tab(children=editors)
for n, (video, frame) in enumerate(samples):
    tabs.set_title(n, f"{video} · {frame}번")
display(tabs)

세 프레임은 검증용입니다. 기록은 `data/annotations/personal_pose_points/`에 프레임별 CSV로 저장됩니다. 원본 왼쪽 위가 (0, 0)이며, `x_px`는 오른쪽으로, `y_px`는 아래로 갈수록 커집니다. 판독 불가 관절은 위치 오차와 그 점이 필요한 각도 계산에서 제외하고 개수를 별도로 셉니다.

판독을 마친 뒤 [자세 추정 베이스라인](../experiments/18_pose_baseline.ipynb)에서 같은 프레임의 수동 좌표와 모델 예측을 비교할 수 있습니다.